# HW3: Atomic cashflows and the Visitor pattern

This assignment develops two skills:

1. **Model cashflow contracts.** Implement `ProductFixedAccruedCashflow` and `ProductOvernightIndexCashflow` using the provided `Product` and `ProductCashflow` bases. Understand their contractual fields, including dates, notional, currency, accrual conventions, index and spread. Each object is a building block for a larger product, such as a swap.
2. **Use the Visitor pattern.** Collect and display each product's attributes through a visitor. This demonstrates how to add an operation without changing the product classes.

Read [IV. Visitor Pattern](https://immense-chocolate-92e.notion.site/IV-Visitor-Pattern-2d4831d0b1d58191bc36ebae3d771d58); implement `accept()` and the display visitor to produce the tables below.

Complete the six `#TODO` blocks in `fixedincomelib/product/linear_products.py` and `fixedincomelib/product/product_display_visitor.py` before running the examples.

Use the provided `qfCreateProductxxx` APIs to create product objects and `qfDisplayProduct` to inspect their fields.

These objects also form a foundation for a later `value_engine`, which can combine their contract attributes with the necessary rates and market data to calculate cash amounts and present values. The engine reads the product objects directly; the display tables are for inspection.

In [1]:
import sys
import QuantLib as ql

from fixedincomelib import (
    qfCreateProductFixedAccruedCashflow,
    qfCreateProductOvernightIndexCashflow,
    qfDisplayProduct,
)
from fixedincomelib.product import (
    ProductCashflow,
    ProductFixedAccruedCashflow, ProductOvernightIndexCashflow,
)

print(f"Python {sys.version.split()[0]} | QuantLib {ql.__version__}")

Python 3.11.16 | QuantLib 1.41


## 1. Fixed-accrual cashflow

The API returns a `ProductFixedAccruedCashflow` object containing contract fields and an accrual year fraction. `qfDisplayProduct` displays its fields; the fraction is checked below. This class has no fixed-rate field.

In [2]:
fixed = qfCreateProductFixedAccruedCashflow(
    effective_date="2026-01-02",
    termination_date="2026-04-02",
    currency="USD",
    notional=1_000_000,
    accrual_basis="ACT/360",
    payment_date="2026-04-06",
    business_day_convention="F",
    holiday_convention="USGS",
)
assert isinstance(fixed, ProductCashflow)
assert isinstance(fixed, ProductFixedAccruedCashflow)
print("Created:", type(fixed).__name__)
qfDisplayProduct(fixed)

Created: ProductFixedAccruedCashflow


,Name,Value
0,Product Type,PRODUCT_FIXED_ACCRUED
1,Notional,1000000
2,Currency,USD
3,Long Or Short,LONG
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,Accrual Basis,ACT/360
7,Payment Date,2026-04-06
8,Business Day Convention,F
9,Holiday Convention,USGS


In [3]:
# 90 calendar days / 360. This number is a year fraction, not a cash amount.
assert fixed.accrued == 90 / 360
assert fixed.termination_date.ISO() == "2026-04-02"
assert fixed.payment_date.ISO() == "2026-04-06"
assert fixed.first_date == fixed.effective_date
assert fixed.last_date == fixed.payment_date
print(f"Accrual year fraction: {fixed.accrued:.6f}")
print("Contractual accrual end:", fixed.termination_date.ISO())
print("Payment date:", fixed.payment_date.ISO())

Accrual year fraction: 0.250000
Contractual accrual end: 2026-04-02
Payment date: 2026-04-06


## 2. Overnight-index cashflow

The API returns a `ProductOvernightIndexCashflow` object. It resolves the tenor to a termination date, obtains currency from the index, and stores the contract fields.

`0.001` is 10 basis points as an annual decimal rate. Negative notional implies `SHORT`. Creation and display do not calculate an interest amount.

In [4]:
overnight = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert isinstance(overnight, ProductCashflow)
assert isinstance(overnight, ProductOvernightIndexCashflow)
assert overnight.notional == -1_000_000
assert overnight.currency.value_str == "USD"
print("Created:", type(overnight).__name__)
qfDisplayProduct(overnight)

Created: ProductOvernightIndexCashflow


,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_CASHFLOW
1,Notional,-1000000
2,Currency,USD
3,Long Or Short,SHORT
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,ON Index,SOFR-1B
7,Compounding Method,COMPOUND
8,Spread,0.001
9,Payment Date,2026-04-06


In [5]:
# Date and tenor inputs resolve to the same termination date.
overnight_by_date = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="2026-04-02",
    overnight_index="SOFR-1B",
    notional=-1_000_000,
    compounding_method="compound",
    spread=0.001,
    payment_date="2026-04-06",
)
assert overnight.termination_date == overnight_by_date.termination_date
assert overnight.payment_date == overnight_by_date.payment_date
print("Date and tenor inputs agree:", overnight.termination_date.ISO())

Date and tenor inputs agree: 2026-04-02


## 3. Currency and payment defaults

Supported indices are `SOFR-1B`, `FF-1B` and `SONIA-1B` (case-insensitive). Currency comes from the index, and an omitted payment date defaults to termination. Compounding labels are `simple`, `arithmetic` and `compound`.

In [6]:
sonia = qfCreateProductOvernightIndexCashflow(
    effective_date="2026-01-02",
    term_or_termination_date="3M",
    overnight_index="SONIA-1B",
    notional=500_000,
    compounding_method="arithmetic",
)
assert sonia.currency.value_str == "GBP"
assert sonia.payment_date == sonia.termination_date
qfDisplayProduct(sonia)

,Name,Value
0,Product Type,PRODUCT_OVERNIGHT_INDEX_CASHFLOW
1,Notional,500000
2,Currency,GBP
3,Long Or Short,LONG
4,Effective Date,2026-01-02
5,Termination Date,2026-04-02
6,ON Index,SONIA-1B
7,Compounding Method,ARITHMETIC
8,Spread,0.0
9,Payment Date,2026-04-02


## 4. Implementation notes and validation

The six `#TODO` blocks are implemented as follows.

* **TODO 1** (`ProductFixedAccruedCashflow.__init__`): calls `ProductCashflow.__init__` with the payment date, which falls back to the termination date when it is `None`, `""` or the null date. It stores the contractual dates, accrual basis and conventions (default `F` / `USGS`), sets `first_date` to the effective date, and computes `accrued` with the `accrued()` date helper. The helper rolls the accrual end for the calculation only, so the stored termination date stays contractual.
* **TODO 2** (`ProductOvernightIndexCashflow.__init__`): looks the index up in `IndexRegistry`. A tenor is advanced with `add_period` on the index's fixing calendar, business-day convention and end-of-month flag; an explicit termination date is kept as given. Both are checked with `_valid_date` / `_check_accrual_dates`. Currency comes from `index.currency()`, and the upper-case index key is kept for display and `serialize()`.
* **TODO 3 / 4** (`accept`): `return visitor.visit(self)`. This is the double dispatch: the product picks the visitor method, and `singledispatchmethod` picks the handler registered for the product's class.
* **TODO 5 / 6** (display handlers): `_common_items` fills the four shared rows, then the product-specific rows are appended in the order given in the TODO. Dates are shown as ISO strings, conventions by their input labels, and the compounding method upper-cased as in `serialize()`.

The cells below check behaviour that the examples above don't cover.

### 4.1 Visitor dispatch

In [7]:
from fixedincomelib.product import Product, ProductDisplayVisitor


class UnregisteredProduct(Product):
    """A product type the display visitor has no handler for."""
    _product_type = "PRODUCT_UNREGISTERED"

    def accept(self, visitor):
        return visitor.visit(self)

    def serialize(self):
        return {}

    @classmethod
    def deserialize(cls, input_dict):
        return cls()


# accept() hands the product to the visitor; the visitor collects rows and display() builds the table.
visitor = ProductDisplayVisitor()
fixed.accept(visitor)
assert visitor.display().equals(qfDisplayProduct(fixed))
assert len(qfDisplayProduct(fixed)) == len(qfDisplayProduct(overnight)) == 10

# A visitor can be reused: each visit starts from a fresh row list.
overnight.accept(visitor)
assert visitor.display().equals(qfDisplayProduct(overnight))

try:
    qfDisplayProduct(UnregisteredProduct())
    raise AssertionError("expected NotImplementedError")
except NotImplementedError as err:
    print("Unregistered product:", err)

Unregistered product: No display visitor for PRODUCT_UNREGISTERED


### 4.2 Serialization round trip

In [8]:
import pandas as pd

rows = []
for name, product in [("fixed", fixed), ("overnight", overnight), ("sonia", sonia)]:
    data = product.serialize()
    rebuilt = type(product).deserialize(data)
    assert rebuilt.serialize() == data
    assert qfDisplayProduct(rebuilt).equals(qfDisplayProduct(product))
    rows.append({"product": name, "TYPE": data["TYPE"], "fields": len(data), "round trip": "ok"})
assert ProductFixedAccruedCashflow.deserialize(fixed.serialize()).accrued == fixed.accrued
pd.DataFrame(rows)

,product,TYPE,fields,round trip
0,fixed,PRODUCT_FIXED_ACCRUED,10,ok
1,overnight,PRODUCT_OVERNIGHT_INDEX_CASHFLOW,9,ok
2,sonia,PRODUCT_OVERNIGHT_INDEX_CASHFLOW,9,ok


### 4.3 Calendars: accrual-end roll, tenor roll and explicit dates

* 2026-01-19 is Martin Luther King Jr. Day. With `F` / `USGS` the accrual end rolls to Tuesday 2026-01-20 for the year fraction (18 days), while `termination_date` and the default payment date stay on the 19th. `business_day_convention="NONE"` gives 17 days.
* 2026-01-03 is a Saturday. `3M` lands on Friday 2026-04-03, which is Good Friday and a SOFR holiday, so the SOFR fixing calendar with `Following` gives Monday 2026-04-06. The same date given explicitly is kept as 2026-04-03.
* SONIA uses the London calendar, where Good Friday and Easter Monday are both holidays, so the same tenor rolls to 2026-04-07.

In [9]:
mlk = qfCreateProductFixedAccruedCashflow("2026-01-02", "2026-01-19", "USD", 1_000_000, "ACT/360")
mlk_unadjusted = qfCreateProductFixedAccruedCashflow(
    "2026-01-02", "2026-01-19", "USD", 1_000_000, "ACT/360", business_day_convention="NONE",
)
assert mlk.business_day_convention.value_str == "F" and mlk.holiday_convention.value_str == "USGS"
assert mlk.termination_date.ISO() == "2026-01-19" and mlk.payment_date.ISO() == "2026-01-19"
assert mlk.accrued == 18 / 360 and mlk_unadjusted.accrued == 17 / 360

sofr_tenor = qfCreateProductOvernightIndexCashflow("2026-01-03", "3M", "SOFR-1B", 1_000_000)
sofr_date = qfCreateProductOvernightIndexCashflow("2026-01-03", "2026-04-03", "SOFR-1B", 1_000_000)
sonia_tenor = qfCreateProductOvernightIndexCashflow("2026-01-03", "3M", "SONIA-1B", 1_000_000)
assert sofr_tenor.termination_date.ISO() == "2026-04-06"
assert sofr_date.termination_date.ISO() == "2026-04-03"
assert sonia_tenor.termination_date.ISO() == "2026-04-07"

pd.DataFrame([
    {"case": "fixed, end on MLK day, F/USGS", "termination": mlk.termination_date.ISO(),
     "payment": mlk.payment_date.ISO(), "accrual days": round(mlk.accrued * 360)},
    {"case": "fixed, end on MLK day, NONE", "termination": mlk_unadjusted.termination_date.ISO(),
     "payment": mlk_unadjusted.payment_date.ISO(), "accrual days": round(mlk_unadjusted.accrued * 360)},
    {"case": "SOFR, 3M from Sat 2026-01-03", "termination": sofr_tenor.termination_date.ISO(),
     "payment": sofr_tenor.payment_date.ISO(), "accrual days": ""},
    {"case": "SOFR, explicit 2026-04-03", "termination": sofr_date.termination_date.ISO(),
     "payment": sofr_date.payment_date.ISO(), "accrual days": ""},
    {"case": "SONIA, 3M from Sat 2026-01-03", "termination": sonia_tenor.termination_date.ISO(),
     "payment": sonia_tenor.payment_date.ISO(), "accrual days": ""},
])

,case,termination,payment,accrual days
0,"fixed, end on MLK day, F/USGS",2026-01-19,2026-01-19,18
1,"fixed, end on MLK day, NONE",2026-01-19,2026-01-19,17
2,"SOFR, 3M from Sat 2026-01-03",2026-04-06,2026-04-06,
3,"SOFR, explicit 2026-04-03",2026-04-03,2026-04-03,
4,"SONIA, 3M from Sat 2026-01-03",2026-04-07,2026-04-07,


### 4.4 Direction, index lookup and input validation

In [10]:
from fixedincomelib.product import LongOrShort

zero = qfCreateProductFixedAccruedCashflow("2026-01-02", "2026-04-02", "USD", 0, "ACT/360")
assert zero.long_or_short == LongOrShort.LONG
assert overnight.long_or_short == LongOrShort.SHORT

fed_funds = qfCreateProductOvernightIndexCashflow("2026-01-02", "3M", "ff-1b", 1_000_000, "simple")
assert fed_funds.currency.value_str == "USD" and fed_funds.on_index.name() == "FedFundsON Actual/360"
assert qfDisplayProduct(fed_funds).set_index("Name").loc["ON Index", "Value"] == "FF-1B"
assert overnight.first_date == overnight.effective_date and overnight.last_date == overnight.payment_date

bad_inputs = {
    "termination before effective": lambda: qfCreateProductFixedAccruedCashflow(
        "2026-04-02", "2026-01-02", "USD", 1, "ACT/360"),
    "overnight end before start": lambda: qfCreateProductOvernightIndexCashflow(
        "2026-04-02", "2026-01-02", "SOFR-1B", 1),
    "unknown index": lambda: qfCreateProductOvernightIndexCashflow("2026-01-02", "3M", "ESTR-1B", 1),
    "unsupported currency": lambda: qfCreateProductFixedAccruedCashflow(
        "2026-01-02", "2026-04-02", "XXX", 1, "ACT/360"),
    "non-finite spread": lambda: qfCreateProductOvernightIndexCashflow(
        "2026-01-02", "3M", "SOFR-1B", 1, spread=float("nan")),
}
for case, build in bad_inputs.items():
    try:
        build()
        raise AssertionError(f"{case}: expected an error")
    except (ValueError, TypeError) as err:
        print(f"{case:30s} -> {type(err).__name__}: {err}")

termination before effective   -> ValueError: termination_date must not precede effective_date
overnight end before start     -> ValueError: termination_date must not precede effective_date
unknown index                  -> ValueError: Unknown overnight index 'ESTR-1B'; supported: SOFR-1B, FF-1B, SONIA-1B
unsupported currency           -> ValueError: Unsupported currency
non-finite spread              -> ValueError: spread must be finite
